# 北辰大学・学生データ 生成ノート（教員用・全14回の土台）

このノートは、統計学Ⅰで**全14回を通して使い回す**架空データ `hokushin_students.csv` を作る。

- 架空の「北辰大学」の学生 400 名分。実在の人物・大学とは無関係。
- 乱数シードを固定しているので、**何度実行しても同じデータ**になる（再現性）。
- 学生にはこの生成ノートは配らない。**CSVだけ**を配る。データの「正解（真の構造）」を学生が知らないからこそ、外れ値・擬似相関を自分で発見できる。

> 末尾に「データ辞書」「真の構造（教員用カンニングペーパー）」「各回での使いどころ」を載せた。各回の模範解答を作るときの参照元。

In [ ]:
import numpy as np
import pandas as pd

# 再現性のため乱数シードを固定（毎回まったく同じデータが生成される）
RNG = np.random.default_rng(2026)
N = 400  # 北辰大学の学生数

## 1. 潜在変数（学生には見えない「真の原因」）

成績やSNS時間の背後には、直接は測れない 2 つの個人差があると仮定する。これがデータの相関構造を生む。

- `discipline`（規律性）… 勉強・睡眠・出席・朝食を増やし、SNSを減らす方向に効く
- `aptitude`（地頭）… テスト点に効く

この 2 つは **CSV には出力しない**。だから学生から見ると「朝食をよく食べる人は成績が良い」のような擬似相関が立ち上がる（第5回の教材）。

In [ ]:
discipline = RNG.normal(0, 1, N)   # 規律性（潜在）
aptitude   = RNG.normal(0, 1, N)   # 地頭（潜在）

## 2. 観測変数の生成

各変数を、潜在変数＋ノイズから作る。係数は「現実にありそうな向きと強さ」で設定してある。

In [ ]:
# --- 学部（3群：第9回 分散分析で使う。テスト点にわずかな差を仕込む） ---
gakubu = RNG.choice(["経済学部", "文学部", "社会福祉学部"], size=N, p=[0.40, 0.35, 0.25])
gakubu_effect = np.select(
    [gakubu == "経済学部", gakubu == "文学部", gakubu == "社会福祉学部"],
    [2.0, -1.0, -1.0],
)

# --- 性別と身長（身長は性別で平均が違う） ---
gender = RNG.choice(["女", "男", "回答しない"], size=N, p=[0.55, 0.43, 0.02])
base_h = np.where(gender == "男", 171.0, 158.0)
base_h = np.where(gender == "回答しない", 165.0, base_h)
height = base_h + RNG.normal(0, 6, N)

# --- 居住形態と通学時間（一人暮らしは大学の近くに住む傾向） ---
lives_alone = RNG.random(N) < 0.35
commute = np.clip(np.where(lives_alone, RNG.normal(20, 8, N), RNG.normal(55, 25, N)), 5, None)

# --- 生活変数（規律性が効く） ---
sleep      = 7 + 0.5 * discipline - 0.005 * (commute - 30) + RNG.normal(0, 0.8, N)
sns        = np.clip(3 - 0.8 * discipline + RNG.normal(0, 1.0, N), 0, None)
study      = np.clip(1.5 + 0.7 * discipline + 0.2 * aptitude + RNG.normal(0, 0.6, N), 0, None)
parttime   = np.clip(np.where(lives_alone, RNG.normal(18, 6, N), RNG.normal(10, 6, N)), 0, None)
attendance = np.clip(82 + 7 * discipline + RNG.normal(0, 5, N), 0, 100)
breakfast  = np.clip(np.round(4 + 1.6 * discipline + RNG.normal(0, 1.0, N)), 0, 7)

# --- テスト点：地頭・勉強・睡眠・SNS・学部で決まる（朝食は直接は効かない！） ---
test = 55 + 7 * aptitude + 4 * study + 1.2 * (sleep - 7) - 1.5 * sns + gakubu_effect + RNG.normal(0, 6, N)
test = np.clip(test, 0, 100)

# --- 世帯年収：右に裾を引く分布（第2・3回 代表値・外れ値で使う） ---
income = RNG.lognormal(mean=np.log(550), sigma=0.45, size=N)

# --- 学生ID ---
ids = [f"26B{i + 1:04d}" for i in range(N)]

In [ ]:
df = pd.DataFrame({
    "学生ID": ids,
    "学部": gakubu,
    "性別": gender,
    "身長cm": np.round(height, 1),
    "一人暮らし": np.where(lives_alone, "はい", "いいえ"),
    "通学時間min": np.round(commute).astype(int),
    "睡眠時間h": np.round(sleep, 1),
    "SNS時間h": np.round(sns, 1),
    "勉強時間h": np.round(study, 1),
    "アルバイト時間week": np.round(parttime).astype(int),
    "出席率": np.round(attendance).astype(int),
    "朝食日数week": breakfast.astype(int),
    "テスト点": np.round(test).astype(int),
    "世帯年収万円": np.round(income).astype(int),
})
df.shape

## 3. わざと「汚す」（現実のデータには必ず傷がある）

外れ値・入力ミス・欠損を意図的に混ぜる。第2回（外れ値）・第4回（前処理・欠損）の教材になる。

In [ ]:
df.loc[3,  "世帯年収万円"] = 12000   # 外れ値：超富裕世帯（平均を吊り上げる）
df.loc[88, "世帯年収万円"] = 9500    # 外れ値：超富裕世帯
df.loc[7,  "身長cm"]     = 1710.0   # 入力ミス：171.0 のつもりが桁ズレ
df.loc[15, "睡眠時間h"]   = np.nan    # 欠損（未記入）
df.loc[42, "睡眠時間h"]   = np.nan    # 欠損
df.loc[101,"通学時間min"] = np.nan    # 欠損
print("汚し完了。欠損のある列:")
print(df.isna().sum()[lambda s: s > 0])

## 4. 中身を確認

In [ ]:
df.head(10)

In [ ]:
df.describe().round(1)

## 5. CSV に保存

In [ ]:
df.to_csv("hokushin_students.csv", index=False, encoding="utf-8-sig")
print("保存しました → hokushin_students.csv")

# Colab からローカルPCにダウンロードしたいとき:
# from google.colab import files
# files.download("hokushin_students.csv")

# Google ドライブに保存したいとき:
# from google.colab import drive
# drive.mount("/content/drive")
# df.to_csv("/content/drive/MyDrive/hokushin_students.csv", index=False, encoding="utf-8-sig")

---
## 付録A：データ辞書（学生にも配ってよい列の説明）

| 列名 | 意味 | 型・単位 |
|---|---|---|
| `学生ID` | 学籍番号（架空） | 文字列 |
| `学部` | 経済学部 / 文学部 / 社会福祉学部 | カテゴリ |
| `性別` | 女 / 男 / 回答しない | カテゴリ |
| `身長cm` | 身長 | cm（1件、入力ミスあり） |
| `一人暮らし` | はい / いいえ | カテゴリ |
| `通学時間min` | 片道の通学時間 | 分 |
| `睡眠時間h` | 1日の平均睡眠 | 時間 |
| `SNS時間h` | 1日のSNS利用 | 時間 |
| `勉強時間h` | 1日の自習 | 時間 |
| `アルバイト時間week` | 1週間のバイト | 時間 |
| `出席率` | 全授業に対する出席 | % |
| `朝食日数week` | 朝食を食べた日数 | 日/週（0〜7） |
| `テスト点` | 期末テスト | 点（0〜100） |
| `世帯年収万円` | 世帯年収 | 万円 |

## 付録B：真の構造（教員用カンニングペーパー・学生には見せない）

生成式の要点。模範解答づくりの根拠。

- **潜在変数** `規律性` が、勉強↑・睡眠↑・出席↑・朝食↑・SNS↓ を同時に動かす。
- **テスト点 = 55 + 7×地頭 + 4×勉強 + 1.2×(睡眠−7) − 1.5×SNS + 学部効果 + 誤差**
  - → 勉強・睡眠・SNSは**本物の**因果。朝食は式に入っていない。
- **朝食 → テスト点 は擬似相関**：朝食もテスト点も「規律性」から来ているだけ。勉強・睡眠を統制すると朝食の効果はほぼ 0 に消える（第5回・第10回の核心）。
- **学部効果**：経済 +2.0／文 −1.0／社福 −1.0。差は小さいので、第9回で「ANOVAでギリギリ検出／多重比較に注意」を教えられる。
- **世帯年収**：対数正規＋極端な外れ値2件 → 平均 ≫ 中央値（第2回）。
- **汚れ**：身長に桁ミス1件、睡眠・通学に欠損3件。

## 付録C：各回での使いどころ

| 回 | 使う列／仕掛け |
|---|---|
| 02 代表値 | `世帯年収万円`（平均と中央値が大きくズレる）／外れ値 |
| 03 ばらつき | `テスト点` の分散・標準偏差・分布の形 |
| 04 Python入門 | 読み込み・`describe`・`groupby`・欠損の確認 |
| 05 相関と因果 | `朝食日数week`×`テスト点`（擬似相関）／`勉強時間h`×`テスト点`（本物） |
| 06 標本とCLT | どれかの列を母集団に見立てて標本平均を反復抽出 |
| 07 区間推定 | 母平均を固定し信頼区間を反復作図 |
| 08 t検定 | `一人暮らし`（はい/いいえ）で `睡眠時間h` or `通学時間min` を比較 |
| 09 ANOVA | `学部`（3群）で `テスト点` を比較 |
| 10 回帰 | `勉強時間h`→`テスト点`（単回帰）／睡眠・SNS追加（重回帰） |
| 11 AIC | 説明変数を増減してAIC比較（朝食を入れても改善しない＝擬似相関の確認） |
| 13 PCA | 数値列をまとめて次元削減 |